<a href="https://colab.research.google.com/github/marcehluna/VC/blob/main/Ejercicio_para_entrega_v1_0_(a_refinar).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resolución del ejercicio de reconocimiento de rostros - Variante 1

In [ ]:
pip install python-docx

## Preparación de Datos (si aplica)

### Subtask:
Descargar el conjunto de datos de imágenes de rostros desde la URL proporcionada para el ejercicio de detección de emociones.


In [ ]:
import os

dataset_url = "https://www.kaggle.com/api/v1/datasets/download/samithsachidanandan/human-face-emotions"
output_path = "/content/human-face-emotions.zip"

print(f"Iniciando descarga del dataset desde: {dataset_url}")
!wget -O {output_path} "{dataset_url}"

if os.path.exists(output_path):
    print(f"\nDescarga finalizada. El archivo '{os.path.basename(output_path)}' se ha guardado en '{output_path}'.")
    print(f"Tamaño del archivo: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
else:
    print(f"\nError: El archivo no se pudo descargar. Por favor, asegúrese de tener los permisos correctos o intente la descarga manualmente. Es posible que necesite configurar las credenciales de Kaggle API.")

Extrae el contenido del archivo comprimido

In [ ]:
import zipfile
import os

zip_file_path = '/content/human-face-emotions.zip'
extraction_path = '/content/human-face-emotions/'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

print(f"Extrayendo el archivo '{os.path.basename(zip_file_path)}' en '{extraction_path}'...")

try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print(f"Extracción completada. Los archivos se han guardado en '{extraction_path}'.")
except Exception as e:
    print(f"Error al extraer el archivo ZIP: {e}")


## Preparación de Datos (si aplica)

### Subtask:
Inspeccionar la estructura del dataset extraído para comprender cómo están organizadas las imágenes y las etiquetas emocionales.


In [ ]:
import os

extraction_path = '/content/human-face-emotions/'

print(f"Contenido del directorio de extracción: '{extraction_path}'")

try:
    # List top-level contents
    top_level_contents = os.listdir(extraction_path)
    print("\n--- Contenido de Nivel Superior ---")
    for item in top_level_contents:
        print(item)

    # Walk through the directory to summarize structure and count files
    print("\n--- Resumen de la Estructura de Directorios y Conteo de Imágenes ---")
    for root, dirs, files in os.walk(extraction_path):
        level = root.replace(extraction_path, '').count(os.sep)
        indent = '    ' * level
        print(f'{indent}{os.path.basename(root)}/ ({len(files)} imágenes)')

except Exception as e:
    print(f"Error al inspeccionar el directorio: {e}")

## Preparación de Datos (si aplica)

### Subtask:
Cargar, redimensionar y convertir a escala de grises las imágenes del dataset, y almacenarlas junto con sus etiquetas.


In [ ]:
import cv2
import os
import numpy as np

# 2. Define la ruta base donde se extrajeron las imágenes
base_path = '/content/human-face-emotions/Data/'

# 3. Define una dimensión de imagen objetivo para el redimensionamiento
target_dim = (64, 64)

# 4. Inicializa dos listas vacías
processed_images = []
labels = []

print(f"Iniciando el preprocesamiento de imágenes desde: {base_path}")
print(f"Redimensionando imágenes a: {target_dim}")

# 5. Itera sobre cada subdirectorio de emociones
for emotion_folder in os.listdir(base_path):
    emotion_path = os.path.join(base_path, emotion_folder)
    if os.path.isdir(emotion_path):
        print(f"Procesando emoción: {emotion_folder}")
        for image_file in os.listdir(emotion_path):
            image_path = os.path.join(emotion_path, image_file)

            # 6.b. Carga la imagen
            img = cv2.imread(image_path)

            if img is not None:
                # 6.c. Conviértela a escala de grises
                gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

                # 6.d. Redimensiona la imagen
                resized_img = cv2.resize(gray_img, target_dim)

                # 6.e. Añade la imagen preprocesada a la lista
                processed_images.append(resized_img)

                # 6.f. Añade el nombre del directorio de la emoción como etiqueta
                labels.append(emotion_folder)
            else:
                print(f"Advertencia: No se pudo cargar la imagen: {image_path}")

# 7. Convierte las listas de imágenes y etiquetas a arrays de numpy
processed_images_np = np.array(processed_images)
labels_np = np.array(labels)

# 8. Muestra el número total de imágenes cargadas y el número de etiquetas
print(f"\nPreprocesamiento completado.")
print(f"Total de imágenes procesadas: {len(processed_images_np)}")
print(f"Total de etiquetas cargadas: {len(labels_np)}")
print(f"Forma de las imágenes procesadas: {processed_images_np.shape}")
print(f"Primeras 5 etiquetas: {labels_np[:5]}")

## Detectar y Guardar Rostros con sus Etiquetas

### Subtask:
Implementar la detección de rostros en las imágenes preprocesadas (en escala de grises y redimensionadas a 48x48) utilizando un clasificador Haar Cascade de OpenCV, y almacenar solo los rostros detectados, también redimensionados a 48x48, junto con sus etiquetas emocionales.

IMPORTANTE HACER NOTAR QUE EL CLASIFICADOR HAAR CASCADE NO TIENE LA MEJOR PERFORMANCE - MEJORA SI SE REDIMENSIONA A 64x64


In [ ]:
print("Descargando el clasificador Haar Cascade...")
!wget https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml -P /content/
print("Clasificador Haar Cascade descargado.")

In [ ]:
import cv2
import numpy as np
import os

# 1. Carga el clasificador Haar Cascade
face_cascade = cv2.CascadeClassifier('/content/haarcascade_frontalface_default.xml')

# 2. Inicializa las listas para almacenar los rostros detectados y sus etiquetas
detected_faces_list = []
detected_labels_list = []

# 3. Define la dimensión objetivo para los rostros detectados
target_face_dim = (48, 48)

print("Iniciando detección de rostros...")

processed_images_count = 0
faces_detected_count = 0

# 4. Itera a través de las imágenes preprocesadas y sus etiquetas
for i, image in enumerate(processed_images_np):
    label = labels_np[i]

    # Realiza la detección de rostros
    # Ajusta scaleFactor y minNeighbors para un mejor rendimiento o precisión
    faces = face_cascade.detectMultiScale(image, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # Verifica si se detectó al menos un rostro
    if len(faces) > 0:
        # Selecciona el primer rostro detectado (puedes modificar para seleccionar el más grande si hay múltiples)
        (x, y, w, h) = faces[0]

        # Recorta la región del rostro
        cropped_face = image[y:y+h, x:x+w]

        # Redimensiona el rostro recortado a la target_face_dim
        resized_face = cv2.resize(cropped_face, target_face_dim)

        # Agrega la imagen del rostro redimensionado y su etiqueta a las listas
        detected_faces_list.append(resized_face)
        detected_labels_list.append(label)
        faces_detected_count += 1

    processed_images_count += 1
    if processed_images_count % 10000 == 0:
        print(f"Procesadas {processed_images_count} imágenes. Rostros detectados hasta ahora: {faces_detected_count}")

# 5. Convierte las listas a arrays de NumPy
detected_faces_np = np.array(detected_faces_list)
detected_labels_np = np.array(detected_labels_list)

print("Rostros detectados y almacenados.")
print(f"Total de imágenes procesadas: {processed_images_count}")
print(f"Total de rostros detectados: {faces_detected_count}")
print(f"Forma de los rostros detectados: {detected_faces_np.shape}")
print(f"Forma de las etiquetas de rostros detectados: {detected_labels_np.shape}")

## Reducción de Dimensionalidad con PCA

### Subtask:
Aplicar el Análisis de Componentes Principales (PCA) a los rostros detectados para reducir la dimensionalidad de los datos, preparando el conjunto para el entrenamiento del modelo.

ANALIZAR VARIANZA E INDICADORES


In [ ]:
from sklearn.decomposition import PCA

# 1. Remodelar el array detected_faces_np a un formato 2D
# Cada imagen (48x48) se aplanará en un vector 1D (48*48 = 2304 píxeles)
num_faces = detected_faces_np.shape[0]
flattened_faces = detected_faces_np.reshape(num_faces, -1)

print(f"Forma de los rostros aplanados: {flattened_faces.shape}")

# 2. Instanciar un objeto PCA con n_components=150
pca = PCA(n_components=150)

# 3. Ajustar el modelo PCA a los datos de rostros aplanados y transformarlos
pca_transformed_faces = pca.fit_transform(flattened_faces)

# 4. Imprimir la forma de los datos resultantes transformados por PCA
print(f"Forma de los datos después de PCA: {pca_transformed_faces.shape}")

# 5. Imprimir la varianza explicada acumulada
cumulative_explained_variance = pca.explained_variance_ratio_.sum()
print(f"Varianza explicada acumulada con 150 componentes principales: {cumulative_explained_variance:.4f}")


## Preparación del Conjunto de Datos

### Subtask:
Dividir los datos transformados por PCA y sus etiquetas correspondientes en conjuntos de entrenamiento y prueba, permitiendo al usuario seleccionar el porcentaje de datos para el conjunto de prueba.

30% DE LOS DATOS SE USAN COMO DATOS DE PRUEBA

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Define el porcentaje para el conjunto de prueba
test_size = 0.3

print(f"Dividiendo los datos en conjuntos de entrenamiento y prueba con test_size={test_size*100}%...")

# 2. Divide los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    pca_transformed_faces, detected_labels_np, test_size=test_size, random_state=42, stratify=detected_labels_np
)

# 3. Imprime las formas de los conjuntos resultantes
print(f"Forma de X_train: {X_train.shape}")
print(f"Forma de X_test: {X_test.shape}")
print(f"Forma de y_train: {y_train.shape}")
print(f"Forma de y_test: {y_test.shape}")

print("División de datos completada.")

## Entrenamiento del Modelo

### Subtask:
Entrenar un modelo de clasificación con los datos de entrenamiento transformados por PCA (X_train, y_train).


In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. Inicializa un objeto LogisticRegression
# Utilizo un random_state para reproducibilidad y solver='saga' que es adecuado para conjuntos de datos grandes
# y puede manejar L1/L2 regularización. max_iter se establece para asegurar la convergencia.
model = LogisticRegression(random_state=42, solver='saga', max_iter=200)

print("Iniciando el entrenamiento del modelo de clasificación...")

# 2. Entrena el modelo utilizando los datos de entrenamiento X_train y y_train
model.fit(X_train, y_train)

print("Entrenamiento del modelo finalizado.")

**Reasoning**:
The model has been successfully trained. Now, I need to evaluate its performance using the test data to assess how well it generalizes to unseen examples. This involves predicting labels for the `X_test` dataset and then calculating and printing the accuracy score.



LA PRECISION DEL MODELO ES POBRE (57%) - CON IMAGENES EN 64x64 ES PEOR - ANALIZAR A QUE SE DEBE

In [ ]:
from sklearn.metrics import accuracy_score

print("Evaluando el modelo con el conjunto de prueba...")

# 1. Realiza predicciones sobre el conjunto de prueba (X_test)
y_pred = model.predict(X_test)

# 2. Calcula la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)

# 3. Imprime la precisión
print(f"Precisión del modelo en el conjunto de prueba: {accuracy:.4f}")

print("Evaluación del modelo finalizada.")

## Verificación y Presentación de Resultados

### Subtask:
Realizar una evaluación exhaustiva del modelo de clasificación entrenado utilizando el conjunto de prueba. Esto incluye la generación de un informe de clasificación detallado y una matriz de confusión para entender el rendimiento del modelo en cada clase.


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print("Generando informe de clasificación...")
# 1. Genera el informe de clasificación
report = classification_report(y_test, y_pred)
print("--- Informe de Clasificación ---")
print(report)

print("Generando matriz de confusión...")
# 2. Crea y muestra la matriz de confusión
fig, ax = plt.subplots(figsize=(10, 8))
display = ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    cmap=plt.cm.Blues,
    colorbar=True,
    ax=ax
)

ax.set_title("Matriz de Confusión del Modelo de Clasificación de Emociones")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("Evaluación exhaustiva completada.")

PRUEBA CON IMAGENES PROPIAS

In [ ]:
from google.colab import files
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Asegurarse de que las variables del modelo y PCA estén disponibles
# model (LogisticRegression), pca (PCA), face_cascade (CascadeClassifier), target_face_dim (tuple)
# Se asume que est\u00e1n cargadas desde las celdas anteriores.

print("Por favor, sube una imagen de un rostro para analizar.")

uploaded = files.upload()

for fn in uploaded.keys():
    print(f"Usuario ha subido el archivo '{fn}'")

    # Leer la imagen subida
    img_path = fn
    img = cv2.imread(img_path)

    if img is None:
        print(f"Error: No se pudo cargar la imagen {img_path}. Aseg\u00farate de que es un archivo de imagen v\u00e1lido.")
        continue

    # Convertir a escala de grises
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Mostrar la imagen original para referencia
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title('Imagen Original Subida')
    plt.axis('off')
    plt.show()

    print("Detectando rostros en la imagen subida...")
    # Detectar rostros
    faces = face_cascade.detectMultiScale(gray_img, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    if len(faces) == 0:
        print("No se detectaron rostros en la imagen subida. Por favor, intente con otra imagen.")
    else:
        # Tomar el primer rostro detectado (se puede mejorar para el rostro m\u00e1s grande si hay m\u00faltiples)
        (x, y, w, h) = faces[0]

        # Recortar y redimensionar el rostro detectado
        cropped_face = gray_img[y:y+h, x:x+w]
        resized_face = cv2.resize(cropped_face, target_face_dim)

        # Preprocesar el rostro para el modelo
        # Aplanar la imagen y aplicar PCA
        processed_face = resized_face.flatten().reshape(1, -1) # Aplanar y remodelar para un solo sample
        pca_transformed_face = pca.transform(processed_face)

        # Realizar la predicci\u00f3n y obtener probabilidades
        probabilities = model.predict_proba(pca_transformed_face)[0]
        predicted_class_idx = np.argmax(probabilities)
        predicted_emotion = model.classes_[predicted_class_idx]
        confidence = probabilities[predicted_class_idx] * 100

        # Mostrar el rostro detectado y la emoci\u00f3n predicha con confianza
        plt.imshow(resized_face, cmap='gray')
        plt.title(f'Rostro Detectado - Emoci\u00f3n: {predicted_emotion} ({confidence:.2f}%)')
        plt.axis('off')
        plt.show()

        print(f"An\u00e1lisis completado. La emoci\u00f3n detectada es: {predicted_emotion} con una confianza de {confidence:.2f}%")
